#  ⭐ `dim_soc`


In [1]:
%run ../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_principio_ativo_atc

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
df = pd.read_parquet(path) 
df.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

In [4]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
df = pd.read_parquet(path) 
df = df[['REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC']].drop_duplicates().sort_values(by=['SOC', 'HLGT', 'HLT', 'PT', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT']) 
df['PK_SOC'] = np.arange(len(df))
df = df.reset_index(drop=True)
df = df[['PK_SOC', 'SOC', 'HLGT', 'HLT', 'PT', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT']]
df.head() 

,PK_SOC,SOC,HLGT,HLT,PT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,0,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação
1,1,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual
2,2,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada
3,3,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação
4,4,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato


### Quality

In [5]:
df.head()

,PK_SOC,SOC,HLGT,HLT,PT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,0,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,Amamentação
1,1,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,Abstinência sexual
2,2,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,Atividade sexual aumentada
3,3,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,Não consumação
4,4,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,Neonato


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18549 entries, 0 to 18548
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   PK_SOC                          18549 non-null  int64 
 1   SOC                             18548 non-null  object
 2   HLGT                            18548 non-null  object
 3   HLT                             18548 non-null  object
 4   PT                              18548 non-null  object
 5   REACAO_EVTO_ADVERSO_MEDDRA_LLT  18548 non-null  object
dtypes: int64(1), object(5)
memory usage: 869.6+ KB


In [7]:
dim = df[['PK_SOC', 'SOC', 'HLGT', 'HLT', 'PT', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT']].value_counts().reset_index(name='COUNT')
dim.query("COUNT > 1").head()


,PK_SOC,SOC,HLGT,HLT,PT,REACAO_EVTO_ADVERSO_MEDDRA_LLT,COUNT


In [8]:
df.SOC.isnull().sum()

np.int64(1)

In [9]:
df.REACAO_EVTO_ADVERSO_MEDDRA_LLT.isnull().sum()

np.int64(1)

In [10]:
df.PT.isnull().sum()

np.int64(1)

# 🥈 Silver

normalized data, medium quality


## hist_atc

In [11]:
hst_soc = df.copy()

In [12]:
hst_soc.query("REACAO_EVTO_ADVERSO_MEDDRA_LLT=='CEFALÉIA'").head()

,PK_SOC,SOC,HLGT,HLT,PT,REACAO_EVTO_ADVERSO_MEDDRA_LLT


In [13]:
hst_soc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18549 entries, 0 to 18548
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   PK_SOC                          18549 non-null  int64 
 1   SOC                             18548 non-null  object
 2   HLGT                            18548 non-null  object
 3   HLT                             18548 non-null  object
 4   PT                              18548 non-null  object
 5   REACAO_EVTO_ADVERSO_MEDDRA_LLT  18548 non-null  object
dtypes: int64(1), object(5)
memory usage: 869.6+ KB


In [17]:
hst_soc.to_parquet(Path(project_root) / "data/02_silver/hst_soc/hst_soc.parquet", index=False)

# 🥇 Gold



## dim_atc

In [16]:
dim_soc = hst_soc.copy()
dim_soc.to_parquet(Path(project_root) / "data/03_gold/dim_soc/dim_soc.parquet", index=False)